
# Hello QuantLib and QuantExt

This dashboard demonstrates that the VREAnalytics Python wrapper constains the lower level quant libraries QuantLib and QuantExt.

Prerequisites: 
- Python 3.7 or newer
- VRE Python module: pip install vannarho-risk-engine

### Load the VRE Python module

## Environment and prerequisites

- Python 3.9–3.13
- Install the self-contained VRE wheel (bundles native deps): `pip install --upgrade vannarho-risk-engine` or point pip at a local `.whl` under `wheelhouse/` or `build/wheel/`.
- No repo-built binaries or `PYTHONPATH` overrides are needed for VRE; the wheel already includes QuantLib/QuantExt/VRE libraries.


In [ ]:
# Wheel bootstrap: prefer installed wheel, fallback to local wheel or PyPI if needed
import sys
import subprocess
from pathlib import Path


def _in_repo_build(mod_path: Path) -> bool:
    parts = mod_path.resolve().parts
    return 'build' in parts and 'VREPython' in parts


def _candidate_roots():
    here = Path.cwd()
    roots = [here]
    roots.extend(list(here.parents)[:3])
    for base in list(roots):
        roots.append(base / 'wheelhouse')
        roots.append(base / 'build' / 'wheel')
    return [r for r in roots if r.exists()]


def _pick_local_wheel():
    candidates = []
    for root in _candidate_roots():
        if root.is_file() and root.suffix == '.whl':
            candidates.append(root)
        elif root.is_dir():
            candidates.extend(root.rglob('*.whl'))
    if not candidates:
        return None
    return max(candidates, key=lambda p: p.stat().st_mtime)


needs_install = False
existing_path = None
try:
    import VRE as _vre  # type: ignore
    existing_path = Path(_vre.__file__)
    needs_install = _in_repo_build(existing_path)
except Exception:
    needs_install = True

if needs_install:
    wheel = _pick_local_wheel()
    source = str(wheel) if wheel else 'vannarho-risk-engine'
    print(f"Installing VRE from {source} ...")
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--upgrade', source])
else:
    print(f"VRE already available: {existing_path}")


In [ ]:
# Verify VRE comes from a wheel install (site-packages)
from pathlib import Path
import importlib.metadata as _md
import VRE as vre  # type: ignore

vre_path = Path(vre.__file__).resolve()
print('VRE module path:', vre_path)
try:
    print('vannarho-risk-engine version:', _md.version('vannarho-risk-engine'))
except _md.PackageNotFoundError:
    print('vannarho-risk-engine distribution not found (using dev build?)')

if 'build' in vre_path.parts and 'VREPython' in vre_path.parts:
    raise RuntimeError('VRE is being imported from a build tree; install the packaged wheel instead.')


In [ ]:
from VRE import *

### Access some QuantLib functionality and set some global data

In [ ]:
today = Settings.instance().evaluationDate 
print ("\ntoday's date is %s" % today.ISO())

calendar = TARGET()
valuationDate = Date(4, October, 2018);
Settings.instance().evaluationDate = valuationDate
print ("valuation date is %s" % valuationDate.ISO())

### Define the (QuantExt) Commodity Forward instrument and related market data

In [ ]:
# Instrument 
name = "Natural Gas";
currency = GBPCurrency();
strikePrice = 100.0;
quantity = 200.0;
position = Position.Long;
maturityDate = Date(4, October, 2022);

# Market
dates = [ Date(20,12,2018),
          Date(20,12,2022) ]
quotes = [ QuoteHandle(SimpleQuote(102.0)),
           QuoteHandle(SimpleQuote(102.0)) ]         
dayCounter = Actual365Fixed()

# Price curve
priceCurve = LinearInterpolatedPriceCurve(valuationDate, dates, quotes, dayCounter, currency);
priceCurve.enableExtrapolation();
priceTermStructure = RelinkablePriceTermStructureHandle();
priceTermStructure.linkTo(priceCurve)

# Discount curve
flatForward = FlatForward(valuationDate, 0.03, dayCounter);
discountTermStructure = RelinkableYieldTermStructureHandle()
discountTermStructure.linkTo(flatForward)

### Set the QuantExt instrument and engine and call the NPV function

In [ ]:
engine = DiscountingCommodityForwardEngine(discountTermStructure)

index = CommoditySpotIndex(name, calendar, priceTermStructure)
instrument = CommodityForward(index, currency, position, quantity, maturityDate, strikePrice);

instrument.setPricingEngine(engine)

print("\nCommodity Forward on '%s' NPV=%.2f %s\n" % (name, instrument.NPV(), instrument.currency().code()))